In [1]:
import os
import re, time, hashlib
from pathlib import Path
from typing import List, Tuple, Dict, Any
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from google import generativeai as genai

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

c:\Users\Lucifer\anaconda3\envs\rag101\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lucifer\anaconda3\envs\rag101\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

INDEX_NAME = "branchindex"
REGION = "us-east-1"
DIM = 384

existing = [d["name"] for d in pc.list_indexes()]
if INDEX_NAME in existing:
    try:
        pc.delete_index(name=INDEX_NAME)
        while INDEX_NAME in [d["name"] for d in pc.list_indexes()]:
            time.sleep(1)
    except Exception as e:
        print("Warning: failed to delete existing index:", e)

pc.create_index(
    name=INDEX_NAME,
    dimension=DIM,
    metric='cosine',
    spec=ServerlessSpec(cloud='aws', region=REGION)
)

# Get a handle to the index
index = pc.Index(INDEX_NAME)

# Namespaces: docs for content; mem_user for long-term memory
DOCS_NS = "docs"
USER_ID = "user_001"
MEM_NS  = f"mem_{USER_ID}"

# Embedding model (384-dim)
EMBED_MODEL = "paraphrase-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

# Gemini LLM
genai.configure(api_key=GOOGLE_API_KEY)
llm = genai.GenerativeModel("gemini-2.5-flash")

In [3]:
TXT_DIR = Path("coffee_txt")
BATCH_SIZE = 16          # tune for your env
CHUNK_SIZE = None        # e.g., 1200 to enable chunking by chars
CHUNK_OVERLAP = 200      # used only if CHUNK_SIZE is set


def read_txt_file(fp: Path) -> Tuple[str, str]:
    """Return (title, body) from a TXT file.
    First non-empty line is treated as title; remaining text as body."""
    raw = fp.read_text(encoding="utf-8")
    lines = [ln.rstrip() for ln in raw.splitlines()]
    # find first non-empty line as title
    title = next((ln for ln in lines if ln.strip()), fp.stem)
    # body is everything after the title's first occurrence
    try:
        first_idx = lines.index(title)
        body = "\n".join(lines[first_idx + 1:]).strip()
    except ValueError:
        body = "\n".join(lines).strip()
    return title.strip(), body


def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    """Simple character-based chunking with overlap."""
    if size is None or size <= 0:
        return [text]
    chunks, i, n = [], 0, len(text)
    step = max(1, size - max(0, overlap))
    while i < n:
        chunks.append(text[i:i+size])
        i += step
    return chunks


def clean_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def make_doc_id(base: str, suffix: str = "") -> str:
    raw = f"{base}:{suffix}" if suffix else base
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


def upsert_txt_dir(
    directory: Path,
    chunk_size: int | None = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
    batch_size: int = BATCH_SIZE,
) -> int:
    """Read *.txt, embed, and upsert into Pinecone."""
    files = sorted(directory.glob("*.txt"))
    if not files:
        print(f"No .txt files found in {directory}")
        return 0

    # Prepare records
    records: List[Tuple[str, str, Dict[str, Any]]] = []
    now = int(time.time())

    for fp in files:
        title, body = read_txt_file(fp)
        content = f"{title}\n\n{body}".strip()
        for j, chunk in enumerate(chunk_text(content, chunk_size, overlap)):
            text_for_embed = clean_spaces(chunk)
            vecs = embedder.encode([text_for_embed], normalize_embeddings=True).tolist()
            doc_id = make_doc_id(fp.stem, str(j) if chunk_size else "")
            meta = {
                "file_name": fp.name,
                "title": title,
                "chunk_idx": j,
                "is_chunked": bool(chunk_size),
                "ts": now,
                "text": text_for_embed
            }
            records.append((doc_id, vecs, meta))

    if records:
        pairs = [(doc_id, vecs[0], meta) for doc_id, vecs, meta in records]
        print(pairs[0][0])
        print(pairs[0][1])  # print first 5 dims of the vector
        print(pairs[0][2])  # print metadata of the first record
        index.upsert(vectors=pairs, namespace=DOCS_NS)
    return len(records) 

# Run
indexed = upsert_txt_dir(TXT_DIR, chunk_size=None)  # set e.g. 1200 to enable chunking
print(f"Indexed {indexed} TXT items into namespace '{DOCS_NS}'.")


62d4b7788eda719c0039adbd84324b90
[-0.053502779453992844, -0.023636149242520332, -0.01574452593922615, 0.04793215170502663, 0.03482086583971977, 0.010648329742252827, 0.032593417912721634, 0.059532925486564636, -0.01560147199779749, -0.03425322845578194, -0.07867628335952759, -0.02786441706120968, -0.08849085122346878, 0.07991019636392593, 0.009445170871913433, -0.05544872581958771, 0.02735068090260029, -0.09651927649974823, 0.05192773416638374, -0.01612941175699234, 0.06399092823266983, -0.04727209359407425, -0.05216621607542038, -0.008339431136846542, 0.0509776696562767, 0.06396114826202393, 0.04701319709420204, 0.09155992418527603, 0.01880245842039585, 0.031828828155994415, 0.022784246131777763, 0.06381214410066605, 0.017639493569731712, -0.018256323412060738, -0.01006863359361887, -0.07532920688390732, 0.04869458079338074, -0.11898058652877808, 0.039980266243219376, 0.025175301358103752, -0.02534213662147522, 0.01041156891733408, 0.030859537422657013, 0.0016110878204926848, -0.01731

In [4]:
import re

def route_query(q: str) -> str:
    ql = q.lower()

    # 1) compare signals
    if re.search(r"\bvs\b|compare|difference between", ql):
        return "compare"

    # 2) recommend/personal signals
    if re.search(r"\bsuggest|recommend|for me|i prefer|i like|what should i try", ql):
        return "recommend"

    # 3) lookup-ish: numbers/ratios/times/temperatures/percentages
    if re.search(r"\b\d+ ?(mg|ml|g|%|minutes|min|ratio|:|°c|celsius)\b", ql):
        return "lookup"

    # 4) default explain
    if re.search(r"\bwhat is|tell me about|benefits of|overview of|explain\b", ql):
        return "explain"

    # safe default
    return "explain"

In [5]:
def embed(text: str):
    return embedder.encode([text], normalize_embeddings=True)[0].tolist()

def retrieve_docs(query_text: str, k: int = 5, keywords=None):
    vec = embed(query_text)
    filt = {"heading_keywords": {"$in": [kw.lower() for kw in keywords]}} if keywords else None
    res = index.query(
        vector=vec, top_k=k, include_values=False, include_metadata=True,
        namespace=DOCS_NS, filter=filt
    )
    return res.get("matches", [])

def retrieve_for_explain(q: str):
    # concise, narrative passages
    return retrieve_docs(q, k=5)

def retrieve_for_compare(q: str):
    # broader context to cover both items
    return retrieve_docs(q, k=10)

def retrieve_for_recommend(q: str):
    # keep doc k modest; personalization will come from memory
    return retrieve_docs(q, k=6)

def retrieve_for_lookup(q: str):
    # try to catch exact tokens that were saved from headings
    # extract simple alpha tokens as "keywords"
    kws = re.findall(r"[A-Za-z][A-Za-z\-]+", q)
    hits = retrieve_docs(q, k=8, keywords=kws[:6])  # cap keyword list
    if not hits:
        # fallback to plain dense
        hits = retrieve_docs(q, k=8)
    return hits    

In [6]:
import yaml, os

TEMPLATE_PATHS = {
    "explain":   "prompts/branch_explain.yaml",
    "compare":   "prompts/branch_compare.yaml",
    "recommend": "prompts/branch_recommend.yaml",
    "lookup":    "prompts/branch_lookup.yaml",
}

def load_template(branch: str) -> str:
    with open(TEMPLATE_PATHS[branch], "r", encoding="utf-8") as f:
        return yaml.safe_load(f)["template"]

In [7]:
def format_history(history):
    if not history: return "None"
    return "\n".join(f"{'User' if h['role']=='user' else 'Assistant'}: {h['content']}" for h in history)

def format_context(hits):
    if not hits: return "None"
    lines = []
    for h in hits:
        doc_id = h.get("id","")
        meta = h.get("metadata",{})
        title = meta.get("title") or meta.get("file_name") or "Untitled"
        text = meta.get("text", "")
        lines.append(f"[{doc_id}] {title} - {text}")
    return "\n".join(lines)

def format_memory(mem_hits):
    if not mem_hits: return "None"
    return "\n".join(f"- {h['metadata'].get('fact','')}" for h in mem_hits)

In [8]:
def build_prompt_explain(query, history, doc_hits):
    tpl = load_template("explain")
    return tpl.format(
        history=format_history(history),
        query=query,
        context_block=format_context(doc_hits)
    )

def build_prompt_compare(query, history, doc_hits):
    tpl = load_template("compare")
    return tpl.format(
        history=format_history(history),
        query=query,
        context_block=format_context(doc_hits)
    )

def build_prompt_recommend(query, history, doc_hits, mem_hits):
    tpl = load_template("recommend")
    return tpl.format(
        history=format_history(history),
        query=query,
        context_block=format_context(doc_hits),
        memory_block=format_memory(mem_hits)
    )

def build_prompt_lookup(query, history, doc_hits):
    tpl = load_template("lookup")
    return tpl.format(
        history=format_history(history),
        query=query,
        context_block=format_context(doc_hits)
    )

In [9]:
# Reuse these from your Memory-RAG notebook if available
def extract_user_facts(txt: str):
    pats = [r"\bI (?:like|love|prefer)\b[^.]+", r"\bI (?:avoid|usually|often|am)\b[^.]+", r"\bMy [A-Za-z ]+\b[^.]+"] 
    out = []
    for pat in pats:
        out += [m.group(0).strip() for m in re.finditer(pat, txt, flags=re.I)]
    return sorted(set(out))

def add_memory_facts(facts: list):
    if not facts: return
    vecs = embedder.encode(facts, normalize_embeddings=True).tolist()
    now = int(time.time())
    payload = []
    for i, fact in enumerate(facts):
        vid = f"{'user_001'}:{now}:{i}"
        payload.append((vid, vecs[i], {"fact": fact, "user_id":"user_001", "ts": now}))
    index.upsert(vectors=payload, namespace=MEM_NS)

def retrieve_memory(q: str, k: int = 3):
    vec = embed(q)
    res = index.query(vector=vec, top_k=k, include_values=False, include_metadata=True, namespace=MEM_NS)
    return res.get("matches", [])

# Global history
history = []

def branched_rag_turn(user_text: str) -> str:
    # 1) branch selection
    branch = route_query(user_text)

    # 2) store new user memory (only from user text)
    add_memory_facts(extract_user_facts(user_text))

    # 3) branch-specific retrieval
    if branch == "explain":
        print("Routing to EXPLAIN branch")
        doc_hits = retrieve_for_explain(user_text)
        prompt = build_prompt_explain(user_text, history, doc_hits)

    elif branch == "compare":
        print("Routing to COMPARE branch")
        doc_hits = retrieve_for_compare(user_text)
        prompt = build_prompt_compare(user_text, history, doc_hits)

    elif branch == "recommend":
        print("Routing to RECOMMEND branch")
        doc_hits = retrieve_for_recommend(user_text)
        mem_hits = retrieve_memory(user_text, k=3)
        prompt = build_prompt_recommend(user_text, history, doc_hits, mem_hits)

    elif branch == "lookup":
        print("Routing to LOOKUP branch")
        doc_hits = retrieve_for_lookup(user_text)
        prompt = build_prompt_lookup(user_text, history, doc_hits)

    else:
        # safe fallback
        print("Routing to default EXPLAIN branch")
        doc_hits = retrieve_for_explain(user_text)
        prompt = build_prompt_explain(user_text, history, doc_hits)

    print("prompt", prompt)  # debug print to check the generated prompt

    # 4) LLM call
    resp = llm.generate_content(prompt)
    answer = getattr(resp, "text", "").strip()

    # 5) update rolling history
    history.append({"role":"user", "content": user_text})
    history.append({"role":"assistant", "content": answer})

    # 6) small trace
    print(f"[Branch: {branch}]  Top docs: {[h['id'] for h in doc_hits[:3]]}")
    print(answer, "\n")
    return answer


In [ ]:
print("Branched RAG -- try queries like:\n"
      " • 'What is espresso coffee?'\n"
      " • 'Compare latte vs mocha'\n"
      " • 'I prefer low-caffeine hot drinks. Suggest something for me'\n"
      "Type 'exit' to quit.\n")

while True:
    q = input("You: ").strip()
    print("query:", q)
    if q.lower() in {"exit", "quit"}:
        print("Goodbye!"); break
    _ = branched_rag_turn(q)

Branched RAG -- try queries like:
 • 'What is espresso coffee?'
 • 'Compare latte vs mocha'
 • 'I prefer low-caffeine hot drinks. Suggest something for me'
Type 'exit' to quit.

query: Why is French press oily?
Routing to EXPLAIN branch
prompt You are a coffee expert. Answer the QUESTION using only CONTEXT. If missing, say:
"I don't know from the provided documents." Cite sources as [DocID].

Conversation so far:
None

QUESTION:
Why is French press oily?

CONTEXT:
[d91e675e5c6811a0fa57c4c930a7c288] French Press - French Press ============ Also known as: Cafetière, Press Pot Origin: France/Italy Brew Method: Immersion brew ~4 minutes; plunge and serve. Ratio: 1:15 to 1:17 Roast: Any Water Temperature: 93–96°C Caffeine: Varies (~100–150 mg per 240 ml) Flavor Notes: full-bodied, oily, rounded Overview: French press emphasizes body and oils, ideal for those who enjoy a richer mouthfeel. FAQs: - Q: Why is French Press oily? A: Metal mesh allows more coffee oils through than paper filters.
[